In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 1. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'train_split.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'val_split.parquet')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# ============================================================
# 2. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols

print(len(wl_cols), len(rain_cols), len(env_cols), len(num_cols))

6 9 7 22


In [3]:
# ============================================================
# 3. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [4]:
# ============================================================
# 4. 기본모델 만들기(주어진 수치형변수만으로, station_id 제외)
# ============================================================

# 4.1 기본모델 독립변수 데이터셋 정의

X_train = train[num_cols]
X_val = val[num_cols]

In [5]:
# 4-2. 기본모델 만들기: LightGBM

from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

model_base = LGBMRegressor(
    random_state = 42
)

model_base.fit(
    X_train,
    y_train
)

pred_base = model_base.predict(X_val)

rmse_base = root_mean_squared_error(
    y_val,  # 기준값(정답)
    pred_base
)

print(f"기본 모델 RMSE: {rmse_base}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027271 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5530
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 22
[LightGBM] [Info] Start training from score 0.111701
기본 모델 RMSE: 0.21560407239403645


In [6]:
# ============================================================
# 5. 모델2: station_id 원핫인코딩 변수 포함 모델링
# ============================================================

# 5-1. 모델2 독립변수 데이터셋 정의

# 원핫인코딩

from sklearn.preprocessing import OneHotEncoder

# 원핫인코더 생성
ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train[['station_id']])
val_st_id_ohe = ohe.transform(val[['station_id']])
# 여기까진 라벨인코더와 동일, 데이터셋에 다시 붙이는게 다름

# train, val셋의 데이터 개수 비교(혹시 train에 없는게 있어서 누락되진 않을지 확인)
# print(len(set(train['station_id'])))
# print(len(set(val['station_id'])))
# print(len(set(train['station_id']) | set(val['station_id'])))
# 각각 데이터 개수와 합집합의 데이터개수가 모두 같음
# 서로 데이터 셋이 같음을 확인

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_with_station = hstack([
    train[num_cols],
    train_st_id_ohe
]).tocsr()  # df + 희소행력(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_with_station = hstack([
    val[num_cols],
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남")
print(X_train.shape, X_train_with_station.shape)
print(X_val.shape, X_val_with_station.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 22개 + 465 = 487이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남
(616331, 22) (616331, 487)
(133325, 22) (133325, 487)


In [7]:
# 5-2. 관측소 ohe 포함 LightGBM

model_with_station = LGBMRegressor(
    random_state = 42
)

model_with_station.fit(
    X_train_with_station,
    y_train
)

pred_with_station = model_with_station.predict(X_val_with_station)

rmse_with_station = root_mean_squared_error(
    y_val,
    pred_with_station
)

print(f"관측소(ohe) 포함 모델 RMSE: {rmse_with_station}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008873 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6460
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 487
[LightGBM] [Info] Start training from score 0.111701
관측소(ohe) 포함 모델 RMSE: 0.21327983252845498
